In [ ]:
def plot_r2(clf,df,var,columns=None):
    df = df.dropna(subset=["ln_profit_loss_before_income_tax_corrected"]).copy()
    if columns is None:
        a = df.drop(columns=[
            "income_tax_paid_on_cash_basis",'ln_profit_loss_before_income_tax_corrected',
            'ln_n_employees',"ln_unrelated_party_revenues","ln_tangible_assets_except_cash"]
            +[_ for _ in df.columns if "fitted" in _]+[_ for _ in df.columns if "wage_monthly" in _])
        columns = np.array(a.columns)[a.dtypes!=object]

    X = df[columns].values
    y = df[var].values

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.4, random_state=42)
    
    #return    X_train, X_test, y_train, y_test  
    
    clf.fit(X_train,y_train)
    profits = clf.predict(X_test)
    try:
        print([_ for _ in sorted(zip(clf.coef_,columns)) if _[0]!=0])
    except:
        pass
    plt.plot(np.exp(y_train),np.exp(clf.predict(X_train)),".",color="lightgrey")
    plt.plot(np.exp(y_test),np.exp(profits),".")
    plt.xscale("log")
    plt.yscale("log")
    plt.title(var)
    plt.xlabel("CbCR")
    plt.ylabel("Predicted")
    
    valid = np.isfinite(y_test) & np.isfinite(profits)
#     print(type(y_test),type(profits))
    r2 = r2_score(y_test[valid],profits[valid])
    plt.title("Var: {} R2: {:.2f}".format(var,r2))

    sns.despine(bottom=True,left=True)

In [ ]:
for var in ['profit_loss_before_income_tax_corrected','n_employees',"unrelated_party_revenues","ln_tangible_assets_except_cash"]:
    est = HistGradientBoostingRegressor(**metaparameters[var])#(alpha=0.03)
    df = foreign_imputation_sample.dropna(subset=[var]).copy()

    a = df.drop(columns=[
        "income_tax_paid_on_cash_basis",'ln_profit_loss_before_income_tax_corrected',
        'ln_n_employees',"ln_unrelated_party_revenues","ln_tangible_assets_except_cash"]
        +[_ for _ in df.columns if "fitted" in _]+[_ for _ in df.columns if "wage_monthly" in _])
    columns = np.array(a.columns)[a.dtypes!=object]


    X = df[columns].values
    y = df[var].values
    a = cross_validate(est, X, y,scoring=scorer,cv=5)
    print(np.mean(a["test_score"]),np.median(a["test_score"]),a["test_score"])

0.9490087801952557 0.9423842562041101 [0.951529   0.92017545 0.99205407 0.94238426 0.93890114]
0.9996991890339272 0.9997879950918441 [0.99987901 0.99933096 0.99966624 0.99983175 0.999788  ]
0.9968568635341167 0.9967003050857374 [0.99549755 0.99670031 0.99824898 0.99653313 0.99730436]
0.9988570025226015 0.9988311848644744 [0.99934812 0.99841028 0.99875444 0.99883118 0.99894099]


In [ ]:
plt.figure(figsize=(14,4))
for i,var in enumerate(['profit_loss_before_income_tax_corrected','n_employees',"tangible_assets_except_cash"]):
    est = HistGradientBoostingRegressor(**metaparameters[var])#(alpha=0.03)
    plt.subplot(1,3,i+1)
    plot_r2(est,cbcr_with_imputed_foreign_values.dropna(subset=[var]),var)

plt.tight_layout()    
plt.savefig(f"{output_figures}/predicted_values_for_foreign_operations.pdf")

KeyError: 'profit_loss_before_income_tax_corrected'

<Figure size 1400x400 with 0 Axes>